# Image classifier for plant leaf diseases.
Through two phases: building a CNN from scratch to understand the mechanics,

then fine-tuning a pretrained model to see what state-of-the-art transfer learning buys you.

- Data pipeline: Use tf.data with ImageDataGenerator.

    Apply augmentation (flip, rotation, zoom) and visualize effects.



---


- Phase 1 : Custom CNN: Design a 4-6 layer CNN with Conv2D → MaxPool → Dropout blocks.

    Train and plot accuracy/loss.



---



- Phase 2 : Transfer Learning: Fine-tune EfficientNetB0 or MobileNetV2.

    Freeze base layers, train head, then unfreeze and fine-tune with a low learning rate.


---


- Visualization: Plot 9 sample predictions with true/predicted labels.

    Show the confusion matrix across all classes.

---

- Analysis: Compare Phase 1 vs Phase 2 accuracy.

    How many parameters does each model have?
    
    What does dropout prevent?



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")

print("Path to dataset files:", path)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import os

dataset_base_path = path 
train_dir = os.path.join(dataset_base_path, 'train')
validation_dir = os.path.join(dataset_base_path, 'validation')
test_dir = os.path.join(dataset_base_path, 'test')


for d in [train_dir, validation_dir, test_dir]:
    if not os.path.exists(d):
        print(f"Warning: Directory '{d}' not found. Please ensure the dataset is downloaded and extracted correctly.")

print(f"Train directory: {train_dir}")
print(f"Validation directory: {validation_dir}")
print(f"Test directory: {test_dir}")

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255, 
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    fill_mode='nearest',
    horizontal_flip=True
    
)

validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical' 
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False 
)

In [ ]:
def plot_images(images_arr):
    fig, axes = plt.subplots(1, 10, figsize=(20, 20))
    axes = axes.flatten()
    for img, ax in zip(images_arr, axes):
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

sample_images, _ = next(train_generator)

print("Original images (first batch):")
plot_images(sample_images[:10])

In [ ]:
from tensorflow.keras import layers, models

num_classes = train_generator.num_classes
print(f"Number of classes: {num_classes}")

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'), 
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5), 
    layers.Dense(512, activation='relu'),
    layers.Dense(num_classes, activation='softmax') 
])

model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
EPOCHS = 15 

history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(EPOCHS)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
print("Evaluating model on test data...")
test_loss, test_acc = model.evaluate(test_generator, steps=test_generator.samples // BATCH_SIZE)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

true_labels = test_generator.classes
predictions = model.predict(test_generator, steps=test_generator.samples // BATCH_SIZE + 1)

predicted_labels = np.argmax(predictions, axis=1)

class_names = list(test_generator.class_indices.keys())

cm = confusion_matrix(true_labels, predicted_labels[:len(true_labels)])

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Custom CNN Model')
plt.show()

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model

base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

base_model.summary()

In [ ]:
global_average_layer = layers.GlobalAveragePooling2D()
dense_layer = layers.Dense(512, activation='relu') 
prediction_layer = layers.Dense(num_classes, activation='softmax')

inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x = base_model(inputs, training=False) 
x = global_average_layer(x)
x = layers.Dropout(0.2)(x) 
x = dense_layer(x)
x = layers.Dropout(0.2)(x) 

outputs = prediction_layer(x)

mobile_net_model = Model(inputs, outputs)

mobile_net_model.summary()

In [ ]:
mobile_net_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
initial_epochs = 10 
history_fine_tune_head = mobile_net_model.fit(
    train_generator,
    epochs=initial_epochs,
    validation_data=validation_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

In [ ]:
base_model.trainable = True
mobile_net_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001), 
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fine_tune_epochs = 10 
total_epochs = initial_epochs + fine_tune_epochs

history_fine_tune = mobile_net_model.fit(
    train_generator,
    epochs=total_epochs,
    initial_epoch=history_fine_tune_head.epoch[-1], 
    validation_data=validation_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

In [ ]:
acc += history_fine_tune.history['accuracy']
val_acc += history_fine_tune.history['val_accuracy']

loss += history_fine_tune.history['loss']
val_loss += history_fine_tune.history['val_loss']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.ylim([0, 1])
plt.plot([initial_epochs-1,initial_epochs-1], plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy (MobileNetV2)')

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.ylim([0, 1.0])
plt.plot([initial_epochs-1,initial_epochs-1], plt.ylim(), label='Start Fine Tuning')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss (MobileNetV2)')
plt.show()

In [ ]:
print("Evaluating fine-tuned MobileNetV2 model on test data...")
test_loss_mobile_net, test_acc_mobile_net = mobile_net_model.evaluate(test_generator, steps=test_generator.samples // BATCH_SIZE)
print(f"\nMobileNetV2 Test Loss: {test_loss_mobile_net:.4f}")
print(f"MobileNetV2 Test Accuracy: {test_acc_mobile_net:.4f}")

In [ ]:
num_samples_to_plot = 9
sample_test_images, sample_test_labels = next(test_generator)
sample_predictions = mobile_net_model.predict(sample_test_images)
predicted_classes = np.argmax(sample_predictions, axis=1)
true_classes = np.argmax(sample_test_labels, axis=1)

class_names_list = list(test_generator.class_indices.keys())

plt.figure(figsize=(12, 12))
for i in range(num_samples_to_plot):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(sample_test_images[i])
    true_label = class_names_list[true_classes[i]]
    predicted_label = class_names_list[predicted_classes[i]]
    color = "green" if predicted_label == true_label else "red"
    plt.title(f"True: {true_label}\nPred: {predicted_label}", color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
true_labels_mobile_net = test_generator.classes

predictions_mobile_net = mobile_net_model.predict(test_generator, steps=len(test_generator))
predicted_labels_mobile_net = np.argmax(predictions_mobile_net, axis=1)

cm_mobile_net = confusion_matrix(true_labels_mobile_net, predicted_labels_mobile_net)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_mobile_net, annot=True, fmt='d', cmap='Blues', xticklabels=class_names_list, yticklabels=class_names_list)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Fine-tuned MobileNetV2 Model')
plt.show()

In [ ]:
print("--- Model Comparison ---")
print(f"Phase 1 (Custom CNN) Test Accuracy: {test_acc:.4f}")
print(f"Phase 2 (Fine-tuned MobileNetV2) Test Accuracy: {test_acc_mobile_net:.4f}")

print("\n--- Model Parameters ---")
print("Custom CNN Model Parameters:")
model.summary()

print("\nFine-tuned MobileNetV2 Model Parameters:")
mobile_net_model.summary()